# EP1 FINAL



In [ ]:
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from google.colab import drive
from tqdm import tqdm

# ==========================================
# 0. AMBIENTE E SEED OFICIAL (123)
# ==========================================

SEED = 123
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Utilizando dispositivo: {device}")

DATA_PATH = "/content/drive/MyDrive/epPLN/train.xlsx"
df = pd.read_excel(DATA_PATH)

def normalizar_texto(texto):
    if not isinstance(texto, str):
        return ""
    texto = re.sub(r'https?://\S+|www\.\S+', 'LINK_URL', texto)
    texto = re.sub(r'(?i)\b(lei|decreto)\s*(?:nº|n°|nº\.|º|°)?\s*\d+[\d\./]*', 'LEI_DECRETO', texto)
    texto = re.sub(r'\b\d{5,}\.\d{6}/\d{4}-\d{2}\b', 'NUMERO_PROCESSO', texto)
    texto = re.sub(r'\(\?\d{2,4}\)?\s*\d{4,5}[-\s]?\d{4}', 'NUMERO_TELEFONE', texto)
    return texto.lower()

df["resp_text_clean"] = df["resp_text"].apply(normalizar_texto)

label2id = {"c1": 0, "c234": 1, "c5": 2}
id2label = {v: k for k, v in label2id.items()}
df["label"] = df["clarity"].map(label2id)

df_train, df_val = train_test_split(df, test_size=0.20, random_state=SEED, stratify=df["label"])

# ==========================================
# 1. MODELO LARGE (335M Parâmetros)
# ==========================================
MODEL_NAME = 'neuralmind/bert-large-portuguese-cased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_LEN = 256

def encode_texts(texts):
    return tokenizer([str(t) for t in texts], padding='max_length', truncation=True, max_length=MAX_LEN, return_tensors="pt")

class TextDataset(Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = torch.tensor(labels, dtype=torch.long) if labels is not None else None

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.encodings['input_ids'])

train_dataset = TextDataset(encode_texts(df_train["resp_text_clean"].to_numpy()), df_train["label"].to_numpy())
val_dataset = TextDataset(encode_texts(df_val["resp_text_clean"].to_numpy()), df_val["label"].to_numpy())

# Batch size 8 para caber na VRAM do Colab
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3).to(device)

# ==========================================
# 2. HIPERPARÂMETROS PARA MODELOS LARGE
# ==========================================
epochs = 4
lr = 1e-5 # Learning rate menor para não desestabilizar as 24 camadas
accum_steps = 4 # Batch efetivo = 8 * 4 = 32

optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
total_steps = (len(train_loader) // accum_steps) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps*0.1), num_training_steps=total_steps)
criterion = nn.CrossEntropyLoss()

# ==========================================
# 3. LOOP DE TREINO E VALIDAÇÃO
# ==========================================
print("\nIniciando Treinamento com BERTIMBAU LARGE...")
best_acc = 0.0

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()

    for step, batch in enumerate(tqdm(train_loader, desc=f"Época {epoch+1}/{epochs} [Treino]")):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = criterion(outputs.logits, labels) / accum_steps
        loss.backward()

        if (step + 1) % accum_steps == 0 or (step + 1) == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

    # Validação
    model.eval()
    preds_all, labels_all = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1)
            preds_all.extend(preds.cpu().numpy())
            labels_all.extend(batch['labels'].numpy())

    acc = accuracy_score(labels_all, preds_all)
    f1_macro = f1_score(labels_all, preds_all, average='macro')
    print(f"\n--> Época {epoch+1} | Val Acc: {acc:.4f} | Val F1-Macro: {f1_macro:.4f}")

    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), 'bertimbau_large_best.pt')
        print(f"    [NOVO RECORD] Modelo LARGE salvo! (Acc: {acc:.4f})")

# ==========================================
# 4. AVALIAÇÃO FINAL DO CAMPEÃO LARGE
# ==========================================
print(f"\n==================================================")
print(f"RESULTADO FINAL DO BERTIMBAU LARGE (Pico: {best_acc:.4f})")
print("==================================================")
model.load_state_dict(torch.load('bertimbau_large_best.pt'))
model.eval()

preds_all, labels_all = [], []
with torch.no_grad():
    for batch in val_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)
        preds_all.extend(preds.cpu().numpy())
        labels_all.extend(batch['labels'].numpy())

print(classification_report(labels_all, preds_all, target_names=["c1", "c234", "c5"], digits=4))

# EP2

Rotulando teste 2 com  BERT ep1 porque  nnão conseguimos melhorar o modelo

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from google.colab import drive

# 1. Montar o Google Drive (se necessário)
drive.mount("/content/drive")

# ==========================================
# 0. AMBIENTE E CAMINHOS
# ==========================================
SEED = 123
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BASE_DIR = Path("/content/drive/MyDrive/epPLN/ep2")
MODEL_EP1_PATH = BASE_DIR / "EP1bertimbau_large_best.pt"
TEST2_PATH = BASE_DIR / "test2.xlsx"
OUTPUT_TEST2_PATH = BASE_DIR / "grupoXX_teste2_rotulado.xlsx"

label2id = {"c1": 0, "c234": 1, "c5": 2}
id2label = {v: k for k, v in label2id.items()}

def normalizar_texto(texto):
    if not isinstance(texto, str):
        return ""
    texto = re.sub(r"https?://\S+|www\.\S+", "LINK_URL", texto)
    texto = re.sub(
        r"(?i)\b(lei|decreto)\s*(?:nº|n°|nº\.|º|°)?\s*\d+[\d\./]*",
        "LEI_DECRETO",
        texto,
    )
    texto = re.sub(r"\b\d{5,}\.\d{6}/\d{4}-\d{2}\b", "NUMERO_PROCESSO", texto)
    texto = re.sub(
        r"\(\?\d{2,4}\)?\s*\d{4,5}[-\s]?\d{4}", "NUMERO_TELEFONE", texto
    )
    return texto.lower()

# ==========================================
# 1. CARREGAR O MODELO PADRÃO DO EP1
# ==========================================
MODEL_NAME = "neuralmind/bert-large-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_LEN = 256

# Arquitetura nativa do Hugging Face usada no EP1
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3
).to(device)

print(f"Carregando os pesos do EP1 de: {MODEL_EP1_PATH}")
model.load_state_dict(torch.load(MODEL_EP1_PATH, map_location=device))
model.eval()
print("✅ Modelo do EP1 carregado com sucesso!")

# ==========================================
# 2. DATASET E DATALOADER PARA O TESTE 2
# ==========================================
class TextDatasetEP1(Dataset):
    def __init__(self, texts):
        self.encodings = tokenizer(
            [str(t) for t in texts],
            padding="max_length",
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt",
        )

    def __getitem__(self, idx):
        return {key: val[idx] for key, val in self.encodings.items()}

    def __len__(self):
        return len(self.encodings["input_ids"])

df_test2 = pd.read_excel(TEST2_PATH)
df_test2["resp_text_clean"] = df_test2["resp_text"].apply(normalizar_texto)

test2_loader = DataLoader(
    TextDatasetEP1(df_test2["resp_text_clean"].to_numpy()),
    batch_size=16,
    shuffle=False,
)

# ==========================================
# 3. INFERÊNCIA E ROTULAGEM DO TESTE 2
# ==========================================
print("\n==========================================")
print("GERANDO ROTULAGEM DO TESTE 2 COM O MODELO DO EP1")
print("==========================================")

preds_test2 = []
with torch.no_grad():
    for batch in test2_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=1)
        preds_test2.extend(preds.cpu().numpy())

# Preenche a coluna 'clarity' com as predições geradas
df_test2["clarity"] = [id2label[p] for p in preds_test2]

# Limpa a coluna temporária de pré-processamento e salva o arquivo final
df_test2_out = df_test2.drop(columns=["resp_text_clean"])
df_test2_out.to_excel(OUTPUT_TEST2_PATH, index=False)

print(f"\n✅ Planilha de entrega gerada com sucesso em: {OUTPUT_TEST2_PATH}")
print(f"Total de instâncias rotuladas: {len(df_test2_out)}")

Mounted at /content/drive


config.json:   0%|          | 0.00/648 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/155 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/210k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

... (3 outputs left out)

[transformers] BertForSequenceClassification LOAD REPORT from: neuralmind/bert-large-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from t

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.34GB            

model.safetensors: downloading bytes:           |  0.00B            

Carregando os pesos do EP1 de: /content/drive/MyDrive/epPLN/ep2/EP1bertimbau_large_best.pt
✅ Modelo do EP1 carregado com sucesso!

GERANDO ROTULAGEM DO TESTE 2 COM O MODELO DO EP1

✅ Planilha de entrega gerada com sucesso em: /content/drive/MyDrive/epPLN/ep2/grupoXX_teste2_rotulado.xlsx
Total de instâncias rotuladas: 897


Regressssao tf idf no teste 1

In [ ]:
from pathlib import Path
import re
import pandas as pd
from google.colab import drive
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score

# ============================================================
# 0. MONTAR O GOOGLE DRIVE E CONFIGURAR CAMINHOS
# ============================================================
drive.mount("/content/drive")

# Ajusta esta pasta de acordo com o teu Drive
BASE_DIR = Path("/content/drive/MyDrive/epPLN/ep2")

PATH_TRAIN = BASE_DIR / "train.xlsx"
PATH_TEST1 = BASE_DIR / "test1-rotulado-para-teste-estatistico.xlsx"


# ============================================================
# 1. FUNÇÃO DE PRÉ-PROCESSAMENTO (MESMO PADRÃO DO EP)
# ============================================================
def normalizar_texto(texto):
    if not isinstance(texto, str):
        return ""
    texto = re.sub(r"https?://\S+|www\.\S+", "LINK_URL", texto)
    texto = re.sub(
        r"(?i)\b(lei|decreto)\s*(?:nº|n°|nº\.|º|°)?\s*\d+[\d\./]*",
        "LEI_DECRETO",
        texto,
    )
    texto = re.sub(r"\b\d{5,}\.\d{6}/\d{4}-\d{2}\b", "NUMERO_PROCESSO", texto)
    texto = re.sub(
        r"\(\?\d{2,4}\)?\s*\d{4,5}[-\s]?\d{4}", "NUMERO_TELEFONE", texto
    )
    return texto.lower()


# ============================================================
# 2. LEITURA E LIMPEZA DOS DADOS
# ============================================================
print(f"Lendo treino em: {PATH_TRAIN}")
df_train = pd.read_excel(PATH_TRAIN)

print(f"Lendo Teste 1 em: {PATH_TEST1}")
df_test1 = pd.read_excel(PATH_TEST1)

df_train["text_clean"] = df_train["resp_text"].apply(normalizar_texto)
df_test1["text_clean"] = df_test1["resp_text"].apply(normalizar_texto)

texto_train = df_train["text_clean"]
y_train = df_train["clarity"]

texto_test1 = df_test1["text_clean"]
y_test1 = df_test1["clarity"]


# ============================================================
# 3. TF-IDF + REGRESSÃO LOGÍSTICA
# ============================================================
vect = TfidfVectorizer()

x_train = vect.fit_transform(texto_train)
x_test1 = vect.transform(texto_test1)

clf = LogisticRegression(class_weight="balanced", random_state=123)
clf.fit(x_train, y_train)


# ============================================================
# 4. AVALIAÇÃO NO TESTE 1
# ============================================================
predicted_test1 = clf.predict(x_test1)

acc = accuracy_score(y_test1, predicted_test1)
f1_macro = f1_score(y_test1, predicted_test1, average="macro")

print("\n==============================================")
print("📊 RESULTADOS BASELINE NO TESTE 1")
print("==============================================")
print(f"Acurácia:  {acc * 100:.2f}%")
print(f"F1 Macro:  {f1_macro:.4f}")
print("----------------------------------------------")
print("\nRelatório de classificação completo:")
print(classification_report(y_test1, predicted_test1))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Lendo treino em: /content/drive/MyDrive/epPLN/ep2/train.xlsx
Lendo Teste 1 em: /content/drive/MyDrive/epPLN/ep2/test1-rotulado-para-teste-estatistico.xlsx

📊 RESULTADOS BASELINE NO TESTE 1
Acurácia:  46.89%
F1 Macro:  0.4680
----------------------------------------------

Relatório de classificação completo:
              precision    recall  f1-score   support

          c1       0.48      0.48      0.48       300
        c234       0.39      0.38      0.39       300
          c5       0.53      0.55      0.54       300

    accuracy                           0.47       900
   macro avg       0.47      0.47      0.47       900
weighted avg       0.47      0.47      0.47       900



/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


melhoria ep1: Para mitigar a saturação de gradiente e capturar melhor o contexto global de respostas mais longas, substituiu-se a extração tradicional baseada apenas no vetor do token especial [CLS] por uma estratégia de Mean Pooling (Média Ponderada das Representações).

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModel
from sklearn.metrics import accuracy_score
from statsmodels.stats.contingency_tables import mcnemar
from google.colab import drive

# 1. Montar Google Drive
drive.mount("/content/drive")

# ==========================================
# 0. CONFIGURAÇÕES E CAMINHOS
# ==========================================
SEED = 123
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BASE_DIR = Path("/content/drive/MyDrive/epPLN/ep2")
TEST1_PATH = BASE_DIR / "test1-rotulado-para-teste-estatistico.xlsx"
PATH_EP1 = BASE_DIR / "EP1bertimbau_large_best.pt"
PATH_GRID2 = BASE_DIR / "bertimbau_large_best_acc.pt"

label2id = {"c1": 0, "c234": 1, "c5": 2}
MODEL_NAME = "neuralmind/bert-large-portuguese-cased"
MAX_LEN = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def normalizar_texto(texto):
    if not isinstance(texto, str):
        return ""
    texto = re.sub(r"https?://\S+|www\.\S+", "LINK_URL", texto)
    texto = re.sub(
        r"(?i)\b(lei|decreto)\s*(?:nº|n°|nº\.|º|°)?\s*\d+[\d\./]*",
        "LEI_DECRETO",
        texto,
    )
    texto = re.sub(r"\b\d{5,}\.\d{6}/\d{4}-\d{2}\b", "NUMERO_PROCESSO", texto)
    texto = re.sub(
        r"\(\?\d{2,4}\)?\s*\d{4,5}[-\s]?\d{4}", "NUMERO_TELEFONE", texto
    )
    return texto.lower()

class TextDataset(Dataset):
    def __init__(self, texts):
        self.encodings = tokenizer(
            [str(t) for t in texts],
            padding="max_length",
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt",
        )

    def __getitem__(self, idx):
        return {key: val[idx] for key, val in self.encodings.items()}

    def __len__(self):
        return len(self.encodings["input_ids"])

# Arquitetura do Grid 2 (Mean Pooling)
class BERTimbauGrid2(nn.Module):
    def __init__(self, model_name, num_classes=3, dropout_rate=0.2):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(512, num_classes),
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden_state = outputs.last_hidden_state

        input_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        sum_embeddings = torch.sum(last_hidden_state * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        mean_pooled = sum_embeddings / sum_mask

        return self.classifier(mean_pooled)

# ==========================================
# 1. CARREGAR DADOS
# ==========================================
df_test1 = pd.read_excel(TEST1_PATH)
df_test1["resp_text_clean"] = df_test1["resp_text"].apply(normalizar_texto)
y_true = df_test1["clarity"].map(label2id).to_numpy()

test1_loader = DataLoader(
    TextDataset(df_test1["resp_text_clean"].to_numpy()),
    batch_size=16,
    shuffle=False,
)

# ==========================================
# 2. CARREGAR MODELO EP1 INTELIGENTE
# ==========================================
print(f"Carregando EP1 de: {PATH_EP1}")
loaded_obj = torch.load(PATH_EP1, map_location=device)

# Caso 1: O arquivo foi salvo diretamente como um objeto nn.Module inteiro
if isinstance(loaded_obj, torch.nn.Module):
    model_ep1 = loaded_obj.to(device)
    model_ep1.eval()

# Caso 2: O arquivo foi salvo via HuggingFace AutoModelForSequenceClassification
elif isinstance(loaded_obj, dict):
    model_ep1 = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=3
    ).to(device)

    state_dict = loaded_obj.get("state_dict", loaded_obj)
    # Ajusta nomes de chaves comuns do Hugging Face / PyTorch Lightning
    new_state_dict = {}
    for k, v in state_dict.items():
        name = k.replace("model.", "").replace("bert_model.", "")
        new_state_dict[name] = v

    model_ep1.load_state_dict(new_state_dict, strict=False)
    model_ep1.eval()

# Rodar inferência do EP1
preds_ep1 = []
with torch.no_grad():
    for batch in test1_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        outputs = model_ep1(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits if hasattr(outputs, "logits") else outputs
        preds = torch.argmax(logits, dim=1)
        preds_ep1.extend(preds.cpu().numpy())
preds_ep1 = np.array(preds_ep1)

# ==========================================
# 3. CARREGAR MODELO GRID 2
# ==========================================
print(f"Carregando Grid 2 de: {PATH_GRID2}")
model_grid2 = BERTimbauGrid2(MODEL_NAME, num_classes=3, dropout_rate=0.2).to(device)
state_g2 = torch.load(PATH_GRID2, map_location=device)
if isinstance(state_g2, dict) and "state_dict" in state_g2:
    state_g2 = state_g2["state_dict"]
model_grid2.load_state_dict(state_g2)
model_grid2.eval()

preds_grid2 = []
with torch.no_grad():
    for batch in test1_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        logits = model_grid2(input_ids, attention_mask)
        preds = torch.argmax(logits, dim=1)
        preds_grid2.extend(preds.cpu().numpy())
preds_grid2 = np.array(preds_grid2)

# ==========================================
# 4. TESTE DE MCNEMAR
# ==========================================
acc_ep1 = accuracy_score(y_true, preds_ep1)
acc_g2 = accuracy_score(y_true, preds_grid2)

correct_ep1 = (preds_ep1 == y_true)
correct_grid2 = (preds_grid2 == y_true)

n11 = np.sum(correct_ep1 & correct_grid2)
n10 = np.sum(correct_ep1 & ~correct_grid2)
n01 = np.sum(~correct_ep1 & correct_grid2)
n00 = np.sum(~correct_ep1 & ~correct_grid2)

contingency_table = [[n11, n10], [n01, n00]]
result = mcnemar(contingency_table, exact=False, correction=True)

print("\n==========================================")
print("📊 RESULTADOS FINAIS DO TESTE DE MCNEMAR")
print("==========================================")
print(f"Acurácia EP1:    {acc_ep1 * 100:.2f}%")
print(f"Acurácia Grid 2: {acc_g2 * 100:.2f}%")
print("\nTabela de Contingência:")
print(f"Ambos Acertaram (n11): {n11}")
print(f"Apenas EP1 Acertou (n10): {n10}")
print(f"Apenas Grid 2 Acertou (n01): {n01}")
print(f"Ambos Erraram (n00): {n00}")
print("------------------------------------------")
print(f"Estatística de Teste (Chi2): {result.statistic:.4f}")
print(f"p-valor: {result.pvalue:.4f}")

if result.pvalue < 0.05:
    print("\nConclusão: A diferença É ESTATISTICAMENTE SIGNIFICATIVA (p < 0.05).")
else:
    print("\nConclusão: NÃO HÁ diferença estatisticamente significativa entre os modelos (p >= 0.05).")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Carregando EP1 de: /content/drive/MyDrive/epPLN/ep2/EP1bertimbau_large_best.pt


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: neuralmind/bert-large-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from t

Carregando Grid 2 de: /content/drive/MyDrive/epPLN/ep2/bertimbau_large_best_acc.pt


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: neuralmind/bert-large-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



📊 RESULTADOS FINAIS DO TESTE DE MCNEMAR
Acurácia EP1:    50.22%
Acurácia Grid 2: 49.44%

Tabela de Contingência:
Ambos Acertaram (n11): 349
Apenas EP1 Acertou (n10): 103
Apenas Grid 2 Acertou (n01): 96
Ambos Erraram (n00): 352
------------------------------------------
Estatística de Teste (Chi2): 0.1809
p-valor: 0.6706

Conclusão: NÃO HÁ diferença estatisticamente significativa entre os modelos (p >= 0.05).


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from google.colab import drive
from tqdm.notebook import tqdm

# 1. Montar o Google Drive
drive.mount("/content/drive")

# ==========================================
# 0. AMBIENTE E DIREITÓRIOS
# ==========================================
SEED = 123
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BASE_DIR = Path("/content/drive/MyDrive/epPLN/ep2")
TRAIN_PATH = BASE_DIR / "train.xlsx"
TEST2_PATH = BASE_DIR / "test2.xlsx"
SAVE_ACC_MODEL_PATH = BASE_DIR / "bertimbau_large_grid_acc.pt"
OUTPUT_TEST2_PATH = BASE_DIR / "grupoXX_teste2_rotulado.xlsx"

label2id = {"c1": 0, "c234": 1, "c5": 2}
id2label = {v: k for k, v in label2id.items()}

def normalizar_texto(texto):
    if not isinstance(texto, str):
        return ""
    texto = re.sub(r"https?://\S+|www\.\S+", "LINK_URL", texto)
    texto = re.sub(
        r"(?i)\b(lei|decreto)\s*(?:nº|n°|nº\.|º|°)?\s*\d+[\d\./]*",
        "LEI_DECRETO",
        texto,
    )
    texto = re.sub(r"\b\d{5,}\.\d{6}/\d{4}-\d{2}\b", "NUMERO_PROCESSO", texto)
    texto = re.sub(
        r"\(\?\d{2,4}\)?\s*\d{4,5}[-\s]?\d{4}", "NUMERO_TELEFONE", texto
    )
    return texto.lower()

# Carregamento do dataset de treino
df_train_full = pd.read_excel(TRAIN_PATH)
df_train_full["resp_text_clean"] = df_train_full["resp_text"].apply(normalizar_texto)
df_train_full["label"] = df_train_full["clarity"].map(label2id)

df_train, df_val = train_test_split(
    df_train_full,
    test_size=0.15,
    random_state=SEED,
    stratify=df_train_full["label"],
)

# ==========================================
# 1. ARQUITETURA COM MEAN POOLING
# ==========================================
MODEL_NAME = "neuralmind/bert-large-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_LEN = 256

class BERTimbauMeanPoolingClassifier(nn.Module):
    def __init__(self, model_name, num_classes=3, dropout_rate=0.2):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size  # 1024

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(512, num_classes),
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden_state = outputs.last_hidden_state

        input_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        sum_embeddings = torch.sum(last_hidden_state * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        mean_pooled = sum_embeddings / sum_mask

        return self.classifier(mean_pooled)

class TextDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.encodings = tokenizer(
            [str(t) for t in texts],
            padding="max_length",
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt",
        )
        self.labels = (
            torch.tensor(labels, dtype=torch.long) if labels is not None else None
        )

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.encodings["input_ids"])

train_loader = DataLoader(
    TextDataset(df_train["resp_text_clean"].to_numpy(), df_train["label"].to_numpy()),
    batch_size=8,
    shuffle=True,
)
val_loader = DataLoader(
    TextDataset(df_val["resp_text_clean"].to_numpy(), df_val["label"].to_numpy()),
    batch_size=16,
    shuffle=False,
)

# ==========================================
# 2. DEFINIÇÃO DA NOVA GRID (FOCO EM ESTABILIDADE E ACC)
# ==========================================
grid_params = [
    {"lr": 5e-6, "dropout": 0.2, "weight_decay": 0.01, "warmup": 0.15},
    {"lr": 8e-6, "dropout": 0.2, "weight_decay": 0.01, "warmup": 0.15},
    {"lr": 1e-5, "dropout": 0.3, "weight_decay": 0.05, "warmup": 0.10},
]

best_acc = 0.0
best_config = None

print("🔍 INICIANDO GRID SEARCH OTIMIZADA PARA ACURÁCIA...")

for config_idx, params in enumerate(grid_params, start=1):
    print(f"\n--- [Teste {config_idx}/{len(grid_params)}] Testando Configuração: {params} ---")

    model = BERTimbauMeanPoolingClassifier(
        MODEL_NAME, num_classes=3, dropout_rate=params["dropout"]
    ).to(device)

    optimizer_grouped_parameters = [
        {"params": [p for n, p in model.bert.named_parameters()], "lr": params["lr"] * 0.8},
        {"params": [p for n, p in model.classifier.named_parameters()], "lr": params["lr"]},
    ]
    optimizer = torch.optim.AdamW(optimizer_grouped_parameters, weight_decay=params["weight_decay"])

    epochs_grid = 3  # 3 épocas por teste para avaliar a convergência real
    accum_steps = 4
    total_steps = (len(train_loader) // accum_steps) * epochs_grid
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(total_steps * params["warmup"]),
        num_training_steps=total_steps,
    )
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs_grid):
        model.train()
        optimizer.zero_grad()
        for step, batch in enumerate(tqdm(train_loader, desc=f"Época {epoch+1}/{epochs_grid}")):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels) / accum_steps
            loss.backward()

            if (step + 1) % accum_steps == 0 or (step + 1) == len(train_loader):
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

    # Avaliação por Acurácia
    model.eval()
    preds_all, labels_all = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            logits = model(input_ids, attention_mask)
            preds = torch.argmax(logits, dim=1)
            preds_all.extend(preds.cpu().numpy())
            labels_all.extend(batch["labels"].numpy())

    val_acc = accuracy_score(labels_all, preds_all)
    print(f"Result Configuração {config_idx}: Val Acurácia = {val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        best_config = params
        torch.save(model.state_dict(), SAVE_ACC_MODEL_PATH)
        print(f"   🏆 [NOVA MELHOR CONFIG] Salva com Val Acc: {val_acc:.4f}")

print("\n==========================================")
print(f"🏆 MELHOR CONFIGURAÇÃO ENCONTRADA:")
print(best_config)
print(f"Pico de Val Acurácia: {best_acc:.4f}")
print("==========================================")

Mounted at /content/drive


config.json:   0%|          | 0.00/648 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/155 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/210k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

... (11 outputs left out)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: neuralmind/bert-large-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Época 1/3:   0%|          | 0/2135 [00:00<?, ?it/s]

Época 2/3:   0%|          | 0/2135 [00:00<?, ?it/s]

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics import accuracy_score
from google.colab import drive

# 1. Montar o Google Drive
drive.mount("/content/drive")

# ==========================================
# 0. AMBIENTE E CAMINHOS
# ==========================================
SEED = 123
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BASE_DIR = Path("/content/drive/MyDrive/epPLN/ep2")
TEST1_PATH = BASE_DIR / "test1-rotulado-para-teste-estatistico.xlsx"
# Ajuste o nome do arquivo do checkpoint do Grid 2 caso tenha salvo com outro nome
SAVE_GRID2_PATH = BASE_DIR / "bertimbau_large_best_acc.pt"

label2id = {"c1": 0, "c234": 1, "c5": 2}

def normalizar_texto(texto):
    if not isinstance(texto, str):
        return ""
    texto = re.sub(r"https?://\S+|www\.\S+", "LINK_URL", texto)
    texto = re.sub(
        r"(?i)\b(lei|decreto)\s*(?:nº|n°|nº\.|º|°)?\s*\d+[\d\./]*",
        "LEI_DECRETO",
        texto,
    )
    texto = re.sub(r"\b\d{5,}\.\d{6}/\d{4}-\d{2}\b", "NUMERO_PROCESSO", texto)
    texto = re.sub(
        r"\(\?\d{2,4}\)?\s*\d{4,5}[-\s]?\d{4}", "NUMERO_TELEFONE", texto
    )
    return texto.lower()

# ==========================================
# 1. ARQUITETURA COM MEAN POOLING (GRID 2)
# ==========================================
MODEL_NAME = "neuralmind/bert-large-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_LEN = 256

class BERTimbauMeanPoolingClassifier(nn.Module):
    def __init__(self, model_name, num_classes=3, dropout_rate=0.2):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(512, num_classes),
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden_state = outputs.last_hidden_state

        input_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        sum_embeddings = torch.sum(last_hidden_state * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        mean_pooled = sum_embeddings / sum_mask

        return self.classifier(mean_pooled)

class TextDataset(Dataset):
    def __init__(self, texts):
        self.encodings = tokenizer(
            [str(t) for t in texts],
            padding="max_length",
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt",
        )

    def __getitem__(self, idx):
        return {key: val[idx] for key, val in self.encodings.items()}

    def __len__(self):
        return len(self.encodings["input_ids"])

# ==========================================
# 2. CARREGAR TESTE 1 E MODELO
# ==========================================
df_test1 = pd.read_excel(TEST1_PATH)
df_test1["resp_text_clean"] = df_test1["resp_text"].apply(normalizar_texto)
y_true_test1 = df_test1["clarity"].map(label2id).to_numpy()

test1_loader = DataLoader(
    TextDataset(df_test1["resp_text_clean"].to_numpy()),
    batch_size=16,
    shuffle=False,
)

model = BERTimbauMeanPoolingClassifier(MODEL_NAME, num_classes=3, dropout_rate=0.2).to(device)
print(f"Carregando pesos do Grid 2 de: {SAVE_GRID2_PATH}")
model.load_state_dict(torch.load(SAVE_GRID2_PATH, map_location=device))
model.eval()

# ==========================================
# 3. INFERÊNCIA E ACURÁCIA NO TESTE 1
# ==========================================
preds_test1 = []
with torch.no_grad():
    for batch in test1_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        logits = model(input_ids, attention_mask)
        preds = torch.argmax(logits, dim=1)
        preds_test1.extend(preds.cpu().numpy())

acc_grid2_test1 = accuracy_score(y_true_test1, preds_test1)

print("\n==========================================")
print(f"🎯 ACURÁCIA REAL DO GRID 2 (MEAN POOLING) NO TESTE 1")
print("==========================================")
print(f"Acurácia Teste 1: {acc_grid2_test1:.4f} ({acc_grid2_test1 * 100:.2f}%)")

MELHORIA EP1: ajuste manual. otimização de hiperparâmetros ajustando manualmente a learning rate para $1.5e-5$, com a adição de uma cabeça de classificação MLP personalizada (camada linear intermediária de 512 unidades, BatchNorm e Dropout de 0,3 para reduzir overfitting).
## Não é significante

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from google.colab import drive
from tqdm.notebook import tqdm

# 1. Montar o Google Drive
drive.mount("/content/drive")

# ==========================================
# 0. AMBIENTE E SEED
# ==========================================
SEED = 123
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BASE_DIR = Path("/content/drive/MyDrive/epPLN/ep2")
TRAIN_PATH = BASE_DIR / "train.xlsx"
SAVE_BEST_GRID_PATH = BASE_DIR / "bertimbau_large_grid_best.pt"

label2id = {"c1": 0, "c234": 1, "c5": 2}
id2label = {v: k for k, v in label2id.items()}

def normalizar_texto(texto):
    if not isinstance(texto, str):
        return ""
    texto = re.sub(r"https?://\S+|www\.\S+", "LINK_URL", texto)
    texto = re.sub(
        r"(?i)\b(lei|decreto)\s*(?:nº|n°|nº\.|º|°)?\s*\d+[\d\./]*",
        "LEI_DECRETO",
        texto,
    )
    texto = re.sub(r"\b\d{5,}\.\d{6}/\d{4}-\d{2}\b", "NUMERO_PROCESSO", texto)
    texto = re.sub(
        r"\(\?\d{2,4}\)?\s*\d{4,5}[-\s]?\d{4}", "NUMERO_TELEFONE", texto
    )
    return texto.lower()

df_train_full = pd.read_excel(TRAIN_PATH)
df_train_full["resp_text_clean"] = df_train_full["resp_text"].apply(normalizar_texto)
df_train_full["label"] = df_train_full["clarity"].map(label2id)

df_train, df_val = train_test_split(
    df_train_full,
    test_size=0.15,
    random_state=SEED,
    stratify=df_train_full["label"],
)

# ==========================================
# 1. ESTRUTURA DO MODELO COM DROPOUT CONFIGURÁVEL
# ==========================================
MODEL_NAME = "neuralmind/bert-large-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_LEN = 256

class BERTimbauCustomClassifier(nn.Module):
    def __init__(self, model_name, num_classes=3, dropout_rate=0.3):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size  # 1024 no Large

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(512, num_classes),
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        return self.classifier(cls_output)

class TextDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.encodings = tokenizer(
            [str(t) for t in texts],
            padding="max_length",
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt",
        )
        self.labels = (
            torch.tensor(labels, dtype=torch.long) if labels is not None else None
        )

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.encodings["input_ids"])

train_loader = DataLoader(
    TextDataset(df_train["resp_text_clean"].to_numpy(), df_train["label"].to_numpy()),
    batch_size=8,
    shuffle=True,
)
val_loader = DataLoader(
    TextDataset(df_val["resp_text_clean"].to_numpy(), df_val["label"].to_numpy()),
    batch_size=16,
    shuffle=False,
)

# ==========================================
# 2. DEFINIÇÃO DA GRADE (GRID SEARCH)
# ==========================================
grid_params = [
    {"lr": 1e-5, "dropout": 0.2, "weight_decay": 0.01},
    {"lr": 1.5e-5, "dropout": 0.3, "weight_decay": 0.01},
    {"lr": 2e-5, "dropout": 0.3, "weight_decay": 0.05},
]

best_score = 0.0
best_config = None

print("🔍 INICIANDO GRID SEARCH DE HIPERPARÂMETROS...")

for config_idx, params in enumerate(grid_params, start=1):
    print(f"\n--- [Teste {config_idx}/{len(grid_params)}] Testando Configuração: {params} ---")

    model = BERTimbauCustomClassifier(
        MODEL_NAME, num_classes=3, dropout_rate=params["dropout"]
    ).to(device)

    optimizer_grouped_parameters = [
        {"params": [p for n, p in model.bert.named_parameters()], "lr": params["lr"] * 0.8},
        {"params": [p for n, p in model.classifier.named_parameters()], "lr": params["lr"]},
    ]
    optimizer = torch.optim.AdamW(optimizer_grouped_parameters, weight_decay=params["weight_decay"])

    epochs_grid = 2  # 2 épocas para testar rapidamente a viabilidade de cada combinação
    accum_steps = 4
    total_steps = (len(train_loader) // accum_steps) * epochs_grid
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(total_steps * 0.1),
        num_training_steps=total_steps,
    )
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs_grid):
        model.train()
        optimizer.zero_grad()
        for step, batch in enumerate(train_loader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels) / accum_steps
            loss.backward()

            if (step + 1) % accum_steps == 0 or (step + 1) == len(train_loader):
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

    # Avaliação da Configuração
    model.eval()
    preds_all, labels_all = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            logits = model(input_ids, attention_mask)
            preds = torch.argmax(logits, dim=1)
            preds_all.extend(preds.cpu().numpy())
            labels_all.extend(batch["labels"].numpy())

    val_f1 = f1_score(labels_all, preds_all, average="macro")
    print(f"Resultado Configuração {config_idx}: F1-Macro = {val_f1:.4f}")

    if val_f1 > best_score:
        best_score = val_f1
        best_config = params
        torch.save(model.state_dict(), SAVE_BEST_GRID_PATH)

print("\n==========================================")
print(f"🏆 MELHOR CONFIGURAÇÃO ENCONTRADA NO GRID:")
print(best_config)
print(f"Melhor Val F1-Macro: {best_score:.4f}")
print("==========================================")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🔍 INICIANDO GRID SEARCH DE HIPERPARÂMETROS...

--- [Teste 1/3] Testando Configuração: {'lr': 1e-05, 'dropout': 0.2, 'weight_decay': 0.01} ---


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: neuralmind/bert-large-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Resultado Configuração 1: F1-Macro = 0.4505

--- [Teste 2/3] Testando Configuração: {'lr': 1.5e-05, 'dropout': 0.3, 'weight_decay': 0.01} ---


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: neuralmind/bert-large-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Resultado Configuração 2: F1-Macro = 0.4466

--- [Teste 3/3] Testando Configuração: {'lr': 2e-05, 'dropout': 0.3, 'weight_decay': 0.05} ---


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: neuralmind/bert-large-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Resultado Configuração 3: F1-Macro = 0.4439

🏆 MELHOR CONFIGURAÇÃO ENCONTRADA NO GRID:
{'lr': 1e-05, 'dropout': 0.2, 'weight_decay': 0.01}
Melhor Val F1-Macro: 0.4505


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics import accuracy_score, f1_score, classification_report
from google.colab import drive

# 1. Montar o Google Drive (se necessário)
drive.mount("/content/drive")

# ==========================================
# 0. AMBIENTE E CAMINHOS
# ==========================================
SEED = 123
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BASE_DIR = Path("/content/drive/MyDrive/epPLN/ep2")
TEST1_PATH = BASE_DIR / "test1-rotulado-para-teste-estatistico.xlsx"
SAVE_GRID1_PATH = BASE_DIR / "bertimbau_large_grid_best.pt"

label2id = {"c1": 0, "c234": 1, "c5": 2}
id2label = {v: k for k, v in label2id.items()}

def normalizar_texto(texto):
    if not isinstance(texto, str):
        return ""
    texto = re.sub(r"https?://\S+|www\.\S+", "LINK_URL", texto)
    texto = re.sub(
        r"(?i)\b(lei|decreto)\s*(?:nº|n°|nº\.|º|°)?\s*\d+[\d\./]*",
        "LEI_DECRETO",
        texto,
    )
    texto = re.sub(r"\b\d{5,}\.\d{6}/\d{4}-\d{2}\b", "NUMERO_PROCESSO", texto)
    texto = re.sub(
        r"\(\?\d{2,4}\)?\s*\d{4,5}[-\s]?\d{4}", "NUMERO_TELEFONE", texto
    )
    return texto.lower()

# ==========================================
# 1. ARQUITETURA DO GRID 1 (CUSTOM HEAD + CLS)
# ==========================================
MODEL_NAME = "neuralmind/bert-large-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_LEN = 256

class BERTimbauCustomClassifier(nn.Module):
    def __init__(self, model_name, num_classes=3, dropout_rate=0.3):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size  # 1024 no Large

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(512, num_classes),
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        return self.classifier(cls_output)

class TextDataset(Dataset):
    def __init__(self, texts):
        self.encodings = tokenizer(
            [str(t) for t in texts],
            padding="max_length",
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt",
        )

    def __getitem__(self, idx):
        return {key: val[idx] for key, val in self.encodings.items()}

    def __len__(self):
        return len(self.encodings["input_ids"])

# ==========================================
# 2. CARREGAR TESTE 1 E MODELO DO GRID 1
# ==========================================
df_test1 = pd.read_excel(TEST1_PATH)
df_test1["resp_text_clean"] = df_test1["resp_text"].apply(normalizar_texto)
y_true_test1 = df_test1["clarity"].map(label2id).to_numpy()

test1_loader = DataLoader(
    TextDataset(df_test1["resp_text_clean"].to_numpy()),
    batch_size=16,
    shuffle=False,
)

model = BERTimbauCustomClassifier(MODEL_NAME, num_classes=3, dropout_rate=0.3).to(device)

print(f"Carregando pesos do Grid 1 de: {SAVE_GRID1_PATH}")
model.load_state_dict(torch.load(SAVE_GRID1_PATH, map_location=device))
model.eval()

# ==========================================
# 3. INFERÊNCIA E MÉTRICAS NO TESTE 1
# ==========================================
preds_test1 = []
with torch.no_grad():
    for batch in test1_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        logits = model(input_ids, attention_mask)
        preds = torch.argmax(logits, dim=1)
        preds_test1.extend(preds.cpu().numpy())

acc_grid1_test1 = accuracy_score(y_true_test1, preds_test1)
f1_grid1_test1 = f1_score(y_true_test1, preds_test1, average="macro")

print("\n==========================================")
print("🎯 ACURÁCIA REAL DO GRID 1 NO TESTE 1 (900 Instâncias)")
print("==========================================")
print(f"Acurácia Teste 1: {acc_grid1_test1:.4f} ({acc_grid1_test1 * 100:.2f}%)")
print(f"F1-Macro Teste 1: {f1_grid1_test1:.4f}")
print("\nRelatório Detalhado por Classe:")
print(classification_report(y_true_test1, preds_test1, target_names=["c1", "c234", "c5"]))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: neuralmind/bert-large-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Carregando pesos do Grid 1 de: /content/drive/MyDrive/epPLN/ep2/bertimbau_large_grid_best.pt

🎯 ACURÁCIA REAL DO GRID 1 NO TESTE 1 (900 Instâncias)
Acurácia Teste 1: 0.4722 (47.22%)
F1-Macro Teste 1: 0.4721

Relatório Detalhado por Classe:
              precision    recall  f1-score   support

          c1       0.50      0.45      0.48       300
        c234       0.44      0.45      0.44       300
          c5       0.48      0.52      0.50       300

    accuracy                           0.47       900
   macro avg       0.47      0.47      0.47       900
weighted avg       0.47      0.47      0.47       900

